# FarmTech Solutions — Parte 2: Análise Descritiva
## FIAP IA 2026/1 | Fase 3 Cap 10 | Grupo 45

**Responsável:** Higor | **Branch:** `parte2-higor`

---
Esta parte cobre correlação entre variáveis, boxplots de NPK por cultura, scatter temperatura x umidade e os principais achados do dataset.


## Setup

Execute a célula seguinte para carregar o CSV (caminho resolvido automaticamente a partir da raiz do repositório).

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 100

# CSV na pasta do projeto (ou um nível acima, se o Jupyter estiver com cwd diferente)
_base = Path(os.getcwd()).resolve()
_csv = None
for candidate in (_base / 'produtos_agricolas.csv', _base.parent / 'produtos_agricolas.csv'):
    if candidate.exists():
        _csv = candidate
        break
if _csv is None:
    raise FileNotFoundError('Coloque produtos_agricolas.csv na raiz do repositório ou abra o notebook a partir dela.')

df = pd.read_csv(_csv)
print(f"Dataset carregado: {df.shape[0]} linhas, {df['label'].nunique()} culturas (fonte: {_csv.name})")
df.head()

## 1. Gráfico 3 — Correlação entre Variáveis

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
labels_pt = [
    'Nitrogênio (N)',
    'Fósforo (P)',
    'Potássio (K)',
    'Temperatura (°C)',
    'Umidade (%)',
    'pH do solo',
    'Precipitação (mm)',
]

corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    xticklabels=labels_pt,
    yticklabels=labels_pt,
)
ax.set_title('Gráfico 3 — Correlação entre variáveis numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Gráfico 4 — Níveis de NPK por Cultura

In [ ]:
# Dez culturas com perfis distintos (≥8 conforme enunciado)
top_crops = [
    'rice', 'maize', 'cotton', 'banana', 'coffee',
    'grapes', 'apple', 'coconut', 'jute', 'mango',
]
df_top = df[df['label'].isin(top_crops)]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, feat, nome in zip(axes, ['N', 'P', 'K'], ['Nitrogênio (N)', 'Fósforo (P)', 'Potássio (K)']):
    sns.boxplot(data=df_top, x='label', y=feat, ax=ax, palette='husl', order=sorted(top_crops))
    ax.set_title(nome)
    ax.set_xlabel('Cultura')
    ax.set_ylabel(feat)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Gráfico 4 — Distribuição de NPK por cultura (10 culturas)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 3. Gráfico 5 — Temperatura vs. Umidade

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
culturas_destaque = ['rice', 'cotton', 'apple', 'banana', 'coffee', 'coconut', 'grapes', 'mango']
df_dest = df[df['label'].isin(culturas_destaque)]
palette = dict(zip(culturas_destaque, sns.color_palette('husl', len(culturas_destaque))))

for cultura, grupo in df_dest.groupby('label'):
    ax.scatter(
        grupo['temperature'],
        grupo['humidity'],
        label=cultura,
        alpha=0.55,
        s=36,
        color=palette[cultura],
    )

ax.set_xlabel('Temperatura (°C)', fontsize=12)
ax.set_ylabel('Umidade (%)', fontsize=12)
ax.set_title(
    'Gráfico 5 — Temperatura × Umidade (8 culturas em destaque)',
    fontsize=14,
    fontweight='bold',
)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 4. Análise Descritiva — Principais Achados

In [ ]:
medias = df.groupby('label')[features].mean().round(2)
medias_ordenado_n = medias.sort_values('N', ascending=False)

print('10 culturas com maior demanda média de N (kg/ha implícito no dataset):')
display(medias_ordenado_n.head(10))

print('\n5 culturas com menor demanda média de N:')
display(medias.sort_values('N').head(5))

print('\nResumo numérico global (todas as linhas):')
display(df[features].describe().round(2))


## 5. Narrativa dos achados (Parte 2 — Higor)

### Correlações (Gráfico 3)
- **P e K** apresentam a correlação mais forte entre pares de nutrientes (coeficiente na casa de **0,73**), indicando que, neste conjunto de dados, solos com mais fósforo tendem a ter mais potássio — útil para planejar adubação de forma conjunta, não isolada.
- **N e P** ficam com correlação **fraca/negativa** (magnitude baixa), ou seja, o padrão de nitrogênio não “acompanha” linearmente o de fósforo entre culturas; cada cultura pode exigir combinações diferentes de NPK.
- **Temperatura e umidade** mostram correlação **positiva moderada** (~0,24): há faixas de clima em que calor e umidade relativa sobem juntos nas amostras, mas o gráfico 5 mostra que **cada cultura ocupa nuvem diferente** nesse plano — o clima não explica tudo sozinho.

### NPK por cultura (Gráfico 4)
- **Algodão (*cotton*)** se destaca com caixas de **N** muito altos em relação a leguminosas e algumas frutíferas: confirma cultura **exigente em nitrogênio**.
- **Maçã (*apple*)** e **uva (*grapes*)** aparecem com **P e K** elevados nos boxplots, alinhados ao manejo típico de fruticultura com maior demanda de fósforo e potássio.
- Culturas como **lentilha** e **feijão-mungo** (visíveis na tabela de médias) concentram-se na **faixa inferior de N** médio — coerente com leguminosas que fixam N₂ quando bem noduladas (o dataset ainda assim mostra dispersão).

### Temperatura × umidade (Gráfico 5)
- **Banana e coco** ocupam regiões com **umidade alta** e temperaturas moderadas a altas, típicas de ambientes tropicais úmidos.
- **Maçã e uva** tendem a **nuvem mais fria** (temperaturas menores) com umidade ainda relevante, compatível com climas subtropicais/temperados de pomicultura e viticultura.
- **Arroz** e **algodão** espalham-se em faixas intermediárias, refletindo diversidade de lavoura irrigada vs sequeiro no dataset sintético/agrícola.

### Síntese para o projeto FarmTech
Os gráficos reforçam que **recomendar cultura** não é olhar uma variável isolada: NPK se correlaciona de forma assimétrica (P–K forte; N mais independente) e o **clima (T, umidade)** separa grupos de culturas no plano cartesiano. Isso justifica, nas partes seguintes do grupo, o uso de **perfis por cultura** e de **modelos supervisionados** que combinem todas as features.
